# Getting Started With mdfs Over HTTP

This notebook starts a local `mdfs-server`, writes markdown files, reads them back, searches the workspace, and commits the result.

Run it from the repository root. The notebook uses a throwaway data directory under `.demo/notebook-http`.

## 1. Start A Local Server

The HTTP server is the safest integration point for notebooks and agents because it acts as the single writer to the mdfs state file.

In [ ]:
from pathlib import Path
import atexit
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def find_repo_root(start=None):
    """Find the mdfs repo even when Jupyter starts in examples/notebooks."""
    env_root = os.environ.get("MDFS_REPO_ROOT") or os.environ.get("MARKDOWNFS_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())

    start_path = Path(start or Path.cwd()).resolve()
    candidates.extend([start_path, *start_path.parents])

    for candidate in candidates:
        if (candidate / "Cargo.toml").exists() and (candidate / "src").is_dir():
            return candidate

    raise RuntimeError(
        "Could not find the mdfs repository root. "
        "Run the notebook from inside the repo or set MDFS_REPO_ROOT."
    )


def find_cargo():
    """Find Cargo even when the Jupyter kernel has a minimal PATH."""
    candidates = []
    env_cargo = os.environ.get("CARGO")
    if env_cargo:
        candidates.append(Path(env_cargo).expanduser())

    path_cargo = shutil.which("cargo")
    if path_cargo:
        candidates.append(Path(path_cargo))

    candidates.extend([
        Path.home() / ".cargo" / "bin" / "cargo",
        Path("/opt/homebrew/bin/cargo"),
        Path("/usr/local/bin/cargo"),
    ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    raise RuntimeError(
        "Could not find Cargo. Install Rust from https://rustup.rs, "
        "or set CARGO=/absolute/path/to/cargo before running this notebook."
    )


ROOT = find_repo_root()
CARGO = find_cargo()
DATA_DIR = ROOT / ".demo" / "notebook-http"
BASE_URL = "http://127.0.0.1:3000"
print(f"Using mdfs repo at {ROOT}")
print(f"Using cargo at {CARGO}")

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)

env = os.environ.copy()
env.update({
    "MARKDOWNFS_DATA_DIR": str(DATA_DIR),
    "MARKDOWNFS_LISTEN": "127.0.0.1:3000",
})
env["PATH"] = os.pathsep.join([str(Path(CARGO).parent), env.get("PATH", "")])

server = subprocess.Popen(
    [CARGO, "run", "--quiet", "--bin", "mdfs-server"],
    cwd=ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def stop_server():
    if server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server.kill()

atexit.register(stop_server)

def request(method, path, body=None, headers=None):
    url = f"{BASE_URL}{path}"
    data = body.encode("utf-8") if isinstance(body, str) else body
    req = urllib.request.Request(url, data=data, method=method, headers=headers or {})
    try:
        with urllib.request.urlopen(req, timeout=5) as response:
            raw = response.read().decode("utf-8")
            content_type = response.headers.get("content-type", "")
            if "application/json" in content_type:
                return json.loads(raw)
            return raw
    except urllib.error.HTTPError as exc:
        message = exc.read().decode("utf-8")
        raise RuntimeError(f"{method} {path} failed: {exc.code} {message}") from exc

for _ in range(60):
    try:
        print(request("GET", "/health"))
        break
    except Exception:
        if server.poll() is not None:
            output = server.stdout.read() if server.stdout else ""
            raise RuntimeError(f"mdfs-server exited early:\n{output}")
        time.sleep(0.25)
else:
    raise TimeoutError("mdfs-server did not become ready")

## 2. Write Markdown Files

Write markdown files directly. The HTTP API creates missing parent directories for new files.

In [ ]:
readme = """# Notebook Demo

This file was written through the mdfs HTTP API.

TODO: delegate the next draft to an agent.
"""

plan = """# Agent Plan

- Inspect the workspace before writing.
- Save every meaningful result as markdown.
- Commit the state after each handoff.
"""

print(request("PUT", "/fs/notebook-demo/readme.md", readme))
print(request("PUT", "/fs/notebook-demo/agent-plan.md", plan))

## 3. Read, List, Search

In [ ]:
print(request("GET", "/fs/notebook-demo/readme.md"))
print(json.dumps(request("GET", "/fs/notebook-demo/"), indent=2))

query = urllib.parse.urlencode({"pattern": "TODO|agent", "path": "notebook-demo", "recursive": "true"})
print(json.dumps(request("GET", f"/search/grep?{query}"), indent=2))

## 4. Commit And Inspect History

In [ ]:
commit = request(
    "POST",
    "/vcs/commit",
    json.dumps({"message": "notebook getting started demo"}),
    {"Content-Type": "application/json"},
)
print(json.dumps(commit, indent=2))
print(request("GET", "/vcs/status"))
print(json.dumps(request("GET", "/vcs/log"), indent=2))

## 5. Clean Up

In [ ]:
stop_server()
print("server stopped")